In [149]:
##### Cleans FAO production value (primary products) data
# aggregates into crop and livestock groupings for rasters, selects closest year to 2020

import os
import pandas as pd

In [150]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
production = pd.read_csv(f"{cd}/Data/Raw/FAO_production/Value_of_Production_E_All_Data/Value_of_Production_E_All_Data.csv")

commodities = pd.read_csv(f"{cd}/Data/Correspondence_tables/FAO_commodity_groups.csv")

country_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/country_names.csv", encoding="cp1252")

# Set save path
save_path = f"{cd}/Data/Clean/Production/FAO_production_value_clean.csv"


In [151]:
#### Initial clean 

# limit to only reported in thousand USD
production = production[production['Element'] == 'Gross Production Value (current thousand US$)']

# keep only countries in project
production['Area Code (M49)'] = production['Area Code (M49)'].str.lstrip("'").astype(float)
production = production[production['Area Code (M49)'].isin(country_codes['FAO_code'])]

# Drop excess year columns and rename
production = production.drop(columns=[col for col in production.columns if col.endswith('F')])
production = production.rename(columns={col: col.lstrip('Y') for col in production.columns if col.startswith('Y')})

In [152]:
### Fill in missing produciton data using closest year to 2020

# join with commodity codes
production = production.merge(commodities, left_on='Item Code', right_on='FAO_item_code', how='inner')

# identify countries missing data
year_cols = [str(y) for y in range(2000, 2025)]
missing_countries = production.groupby(['Area Code (M49)'])[year_cols].sum().reset_index()

missing_countries = missing_countries[missing_countries['2020'] == 0]

# find most recent year with data for each country
def most_recent_nonzero(row):
    # look backwards from 2019 to find most recent non-zero year
    for year in range(2019, 1999, -1):
        if row[str(year)] != 0:
            return year
    return None

missing_countries['closest_year'] = missing_countries.apply(most_recent_nonzero, axis=1)

# get rid of unimportant info
missing_countries = missing_countries[['Area Code (M49)', 'closest_year']].dropna()

# partition produciton data based on if countries are missing 
production_2020 = production[~production['Area Code (M49)'].isin(missing_countries['Area Code (M49)'])]
production_fill = production[production['Area Code (M49)'].isin(missing_countries['Area Code (M49)'])]

# clean 2020 data
col_to_keep = ['Area Code (M49)', 'Item Code', '2020', 'final_group'] 
production_2020 = production_2020[col_to_keep].dropna()

# fill in missing data for missing countries based on closest available year
production_fill = production_fill.merge(missing_countries, on='Area Code (M49)', how='left')
production_fill['2020'] = production_fill.apply(lambda row: row[str(int(row['closest_year']))], axis=1)

col_to_keep = ['Area Code (M49)', 'Item Code', '2020', 'final_group'] 
production_fill = production_fill[col_to_keep].dropna()

# concatinate back together 
production = pd.concat([production_2020, production_fill], ignore_index=True)

In [153]:
### Sum by commodity group

production_sum = production.groupby(['Area Code (M49)', 'final_group'])['2020'].sum().reset_index()

In [154]:
### final clean 

# add units
production_sum['Unit'] = 'thousand USD (nominal)'

# add ISO codes
production_sum = production_sum.merge(country_codes, left_on='Area Code (M49)', right_on='FAO_code', how='left')

col_to_keep = ['ISO3', 'final_group', 'Unit', '2020'] 
production_sum = production_sum[col_to_keep]

In [155]:
# Save
production_sum.to_csv(save_path, index=False)